# RAG at a glance

We start with a bare-bones RAG pipeline, and then show where evaluation fits.

<img src="https://ecoop.github.io/rulebook/img/flow_overview.svg" alt="The RAG pipeline in two parts. Prepare, once per rulebook: rulebook PDFs go through ingestion, chunking and embeddings into embedding space. Answer, for each question: retrieval finds the nearest chunks in that space, and generation writes a cited answer." width="900" style="display: block; clear: both; max-width: 100%; height: auto">

**In this lesson**

1. **Ingestion**: reading a rulebook into plain text.
2. **Chunking**: cutting it into pieces, one rule each.
3. **Embeddings**: a vector for each chunk.
4. **Retrieval**: the chunks nearest a question.
5. **Generation**: Claude's cited answer.
6. **Evaluation**: where checks fit.

On Colab, run the cells from the top; the first one fetches the repo (about a minute). On your
own machine, run `uv sync`, then `uv run jupyter lab` from the repo root.

This tour is also a place to get comfortable with Jupyter notebooks and experimenting with code:
edit any code cell and rerun it with Shift+Enter. The **Try it** notes suggest where to start.
The text between cells describes the cells as written, so it won't follow your edits.

In [ ]:
# On Colab, fetch the repo first (about a minute). On your own machine this is skipped.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/ecoop/rulebook /content/rulebook
    %pip install -qq -e /content/rulebook
    sys.path.insert(0, "/content/rulebook/src")  # this session can't see the new install otherwise

from rulebook.notebook import setup, show_table
setup()  # work from the repo root and load .env

## 1. Ingestion

<img src="https://ecoop.github.io/rulebook/img/flow_ingestion.svg" alt="Rulebook PDFs are extracted to plain text, page by page." width="520" style="max-width: 100%; height: auto">

**Ingestion** is reading a file, or some other data source, into plain text; `extract_pages`
reads PDF, Markdown or plain text, page by page. For this tour, we ingest two rulebooks, checkers
and tennis, and look at the start of the checkers text:

In [ ]:
from pathlib import Path
from rulebook.ingest import extract_pages

pages = {}
for domain in ['checkers', 'tennis']:
    doc = Path(f'rules/{domain}/{domain}.md')
    pages[domain] = extract_pages(doc)
    print(f'{domain}: {sum(len(p.text) for p in pages[domain]):,} characters from {doc.name}')

print('\n' + pages['checkers'][0].text[:400] + '…')

## 2. Chunking

<img src="https://ecoop.github.io/rulebook/img/flow_chunking.svg" alt="Plain text is split into short chunks." width="520" style="max-width: 100%; height: auto">

**Chunking** breaks the ingested text into smaller pieces, like the individual rules of a game.
RAG retrieves these *pieces*, not whole files, and how they are cut is one of the biggest levers
on the whole system. Here the chunker splits each rulebook on its own rule numbers, so each chunk
is one rule you can cite:

In [ ]:
from rulebook.chunking import chunk_pages

rulebooks = {}
for domain in pages:
    rulebooks[domain] = chunk_pages(pages[domain], source=f'{domain}.md', domain=domain)
    print(f'{domain}: {len(rulebooks[domain])} chunks')

In [ ]:
domain = 'checkers'
n = 5
print(f'The first {n} {domain} chunks:')
show_table(['rule', 'text'], [(c.rule_id, c.text[:80] + '…') for c in rulebooks[domain][:n]])

The `rule` id on each chunk is what an answer cites, so a reader can check a claim against the
actual rule. Text before the first numbered rule becomes a `preamble` chunk.

**Try it.** Set `n = 17` in the code cell above and rerun it to see every checkers chunk, the
`preamble` first. Or set `domain = 'tennis'` to see the tennis chunks.

## 3. Embeddings

<img src="https://ecoop.github.io/rulebook/img/flow_embeddings.svg" alt="An embedding model embeds a chunk as a vector, a list of numbers, stored as a point in embedding space." width="520" style="max-width: 100%; height: auto">

A chunk's **embedding** is a vector computed from its text: a fixed-length list of numbers that
represents what the chunk says. An **embedding model** computes them, placing chunks that say
similar things near each other in its **embedding space**.

The model sets the length. BERT (2018), which much of today's text embedding grew from, made 768
numbers per chunk; hosted models now make about 1,000 to 3,000 (Voyage's default is 1,024, OpenAI's
largest model 3,072). The small local model this tour uses makes 512, as the code below prints.
More numbers can hold finer distinctions, but cost more to store and compare.

Equal length isn't enough to compare two vectors. Each model lays out its space its own way, so a
question has to be embedded by the same model as the chunks it's compared with, and changing the
model means embedding every chunk again. That's why the index records which model built it.

The code below loops over `rulebooks` from above: `rulebooks.items()` gives `(domain, chunks)`
pairs. For each rulebook it embeds the chunks' text and stores the vectors with their chunks,
which together make up that domain's **index** (a few seconds, with a small local model):

In [ ]:
from rulebook.config import settings
from rulebook.embeddings import get_embedder
from rulebook.store import domain_index_path, write_store

embedder = get_embedder()
for domain, chunks in rulebooks.items():
    vectors = embedder.embed([c.text for c in chunks], input_type='document')
    write_store(domain_index_path(settings.resolved_index_path, domain), chunks, vectors,
                provider=settings.embedding_provider, model=settings.embedding_model)
    print(f'{domain}: {len(vectors)} embeddings of {len(vectors[0])} numbers each, stored')

## 4. Retrieval

<img src="https://ecoop.github.io/rulebook/img/flow_retrieval.svg" alt="The question is embedded by the same model, and its nearest chunks in embedding space come back as passages." width="520" style="max-width: 100%; height: auto">

**Retrieval** is finding the chunks that best match a question. The same model computes an
embedding for the question, which places it in that same space, and the chunks nearest to it
come back. **Distance** is 2 − 2 × the cosine similarity of the two vectors: 0 means identical,
2 unrelated, 4 opposite. Lower is nearer.

In [ ]:
from rulebook.retrieve import retrieve

question = 'can a king move backwards?'
hits = retrieve(question, domain='checkers', k=3)
show_table(['rank', 'rule', 'distance', 'text'],
           [(i, h.rule_id, f'{h.distance:.2f}', h.text[:80] + '…') for i, h in enumerate(hits, 1)])

Rule 4.2, *How a king moves*, comes first, though the question barely shares its words.

Retrieval doesn't answer the question. It finds the rules an answer can be written from, and
generation, next, writes the answer from them.

Here each step has its own code, next to the section that explains it. In practice they run
together: `build_index` ingests, chunks, embeds and stores each rulebook in one call, and that
is how the app and the *A closer look at* notebooks build their indexes.

**Go further:** <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/embeddings_retrieval.ipynb" target="_parent">A closer look at Embeddings and Retrieval</a> looks inside an embedding and shows a question where the
nearest chunk isn't the rule that answers it.

**Try it.** Set `question` to your own, change `k`, or set `domain='tennis'`, and rerun the
code cell above. (Generation, next, has saved answers only for this tour's own questions.)

## 5. Generation

<img src="https://ecoop.github.io/rulebook/img/flow_generation.svg" alt="The question and passages go to Claude as a prompt, and Claude generates a cited answer." width="520" style="max-width: 100%; height: auto">

**Generation** is a language model writing the answer. Rulebook hands Claude *only* the
retrieved chunks, and Claude cites each claim by its rule, as `[checkers 4.1]`.
<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/generation.ipynb" target="_parent">A closer look at Generation</a> shows exactly what Claude is sent and
why, and how to set up the API key that asking Claude takes.

So this tour can run without a key, here are Claude's answers from a real run, saved with the
repo. The first is for the same `question` as in Retrieval. The check under each answer runs
now: `find_unverified_citations` lists any rule the answer cites that isn't among the chunks
retrieved for the question.

In [ ]:
import json
from rulebook.generate import find_unverified_citations

saved = json.load(open('notebooks/data/rag_at_a_glance_answers.json'))
answers = {a['question']: a['answer'] for a in saved['answers']}

# Answers are saved only for this tour's own questions; with any other, show the first one.
shown = question
if shown not in answers:
    shown = 'can a king move backwards?'
    print(f"No saved answer for {question!r}: answers are saved only for this tour's own "
          "questions. Asking Claude your own takes an API key; A closer look at Generation "
          f"shows how. Here is the saved answer for {shown!r}.\n")

print(f"Claude ({saved['model']}), recorded {saved['recorded']}:\n")
chunks = retrieve(shown, domain='checkers', k=5)
print(answers[shown])
print('\nunverified citations:', find_unverified_citations(answers[shown], chunks))

**The guardrail.** Asked about something the rules don't contain, Claude says they don't cover
it, instead of answering from what it already knows.

In [ ]:
print(answers['What is the offside rule in soccer?'])

**Two rulebooks at once.** Retrieval can take the nearest chunks from each of several rulebooks,
and Claude compares them.

In [ ]:
from rulebook.retrieve import retrieve_across_domains

question = 'How do you win each game?'
chunks = retrieve_across_domains(question, ['checkers', 'tennis'], k_per_domain=5)
print('retrieved from:', sorted({c.domain for c in chunks}), '\n')
print(answers[question])
print('\nunverified citations:', find_unverified_citations(answers[question], chunks))

## 6. Evaluation

**Evaluation** is measuring how well the pipeline works. It comes last here, but it applies at
different points in the pipeline, and some checks need the right answer known in advance while
others don't:

- **Chunking**: does each chunk hold exactly one rule? That can be checked from the chunks
  alone.
- **Retrieval**: does the chunk that answers a question come back near the top? You can
  check that only if you know which chunk answers; above, you could see by eye that it was
  rule 4.2.
- **Generation**: does the answer stick to its chunks? That can be checked from the answer
  and its chunks alone, as `find_unverified_citations` did for the saved answers. Is the answer right? That needs
  the right answer.

You run an evaluation after changing a stage, to learn whether the change helped. Knowing the
answer by eye works for one question; after each change you'd want to recheck dozens.
<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/embeddings_retrieval.ipynb" target="_parent">A closer look at Embeddings and Retrieval</a> shows where checking by eye stops working, and
<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/evaluation.ipynb" target="_parent">A closer look at Evaluation</a> writes the answers down once, as a **gold set**, and scores against them.

## Where to go next

- **Go further** with the *A closer look at* notebooks, all readable without a key: <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/embeddings_retrieval.ipynb" target="_parent">Embeddings</a> (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/embeddings_retrieval.ipynb" target="_parent">Colab</a>),
  then <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/evaluation.ipynb" target="_parent">Evaluation</a> (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/evaluation.ipynb" target="_parent">Colab</a>),
  then <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/reranking.ipynb" target="_parent">Reranking</a> (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/reranking.ipynb" target="_parent">Colab</a>),
  then <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/chunking.ipynb" target="_parent">Chunking</a> (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/chunking.ipynb" target="_parent">Colab</a>),
  then <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/generation.ipynb" target="_parent">Generation</a> (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/generation.ipynb" target="_parent">Colab</a>), and more of these notebooks as they're added (listed on the <a href="https://ecoop.github.io/rulebook/" target="_parent">notebooks page</a>).
- **Turn a knob.** `retrieve` returns the `k` nearest chunks; rerun the retrieval above with `k=1` or `k=8`
  and see what comes back. Or edit a rule in `rules/checkers/checkers.md`, rebuild, and retrieve again.
- **Run generation live.** With an Anthropic API key, `rulebook.pipeline.ask` retrieves and asks Claude in
  one call, about a cent a question; <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/generation.ipynb" target="_parent">A closer look at Generation</a>
  walks through it.
- **Add a real ruleset.** `uv run python -m rulebook.add <slug> <your.pdf>` ingests and indexes a
  rule document; `rules/SOURCES.md` lists recognizable ones.
- **Read the core modules** in order: `ingest` → `chunking` → `embeddings` → `store` → `retrieve` →
  `generate`. The rest of `src/rulebook/` (auth, roles, cost tracking) is app scaffolding.